# Assignment Template: Adsorption on a Metal Surface
# 作业模板：金属表面的吸附

This notebook is a *scaffold* for your assignment. The first half (bulk metal,
convergence tests, basic surface) has most of the code written for you, with
**blanks** you must fill. The second half is not here at all: that part you
build yourself, using the class notebook `Castep_adsorption_study.ipynb` as
your guide (guide, not source of copy-paste: I will recognise my own code).

这个 notebook 是你作业的"模板"。前半部分（体相金属、收敛测试、基础表面）的
大部分代码已经写好，但留有**空白**需要你填写。后半部分没有在这个模板当中给出：
那部分需要你自己写，可以参考课堂 notebook `Castep_adsorption_study.ipynb`
（这只是参考，不要复制粘贴代码：我认得出我自己写的代码）。

## How the blanks work / 空白怎么填

Every `____` must be replaced by you. An unfilled `____` is a **syntax
error**: the cell (or the job it submits) will fail immediately and the error
message will point at the line. That is deliberate. Read the error, find the
line, think, fill it in.

每个 `____` 都必须由你来填写。没填的 `____` 将会导致**语法错误**：单元格（或它
提交的任务）会立刻报错，错误信息会指出是哪一行导致报错。这是有意为之的设计。
请阅读错误信息，找到那一行，思考，然后填空。

**If a submitted job finishes within seconds, something failed.** Look at the
job output with `/tail <dir>/job_<id>.out` in the bot chat, or open the file
in Jupyter. The error tells you which blank you got wrong.

**如果提交的任务几秒钟就结束了，说明出错了。** 用机器人里的
`/tail <目录>/job_<id>.out` 查看任务输出，或在 Jupyter 里打开这个文件。
错误信息会告诉你哪个空填错了。

## The rules / 规则

* **Not gold, not atomic oxygen.** Choose your own FCC metal (Cu, Ag, Pt,
  Pd, Al) and your own adsorbate (H or N atom, or a small molecule: CO,
  H$_2$O, NH$_3$). The next cell will refuse to continue until you do.
* **不能用金，不能用氧原子。** 选择你自己的 FCC 金属（Cu、Ag、Pt、Pd、Al）
  和你自己的吸附物（H 或 N 原子，或小分子：CO、H₂O、NH₃）。
  下一个单元格在你选好之前不会继续运行。
* Marks are for evidence **and** notes. Write what you did and why in the
  fill-in markdown cells. 评分依据是计算证据和笔记说明。在填空的 Markdown 单元格里写下你做了什么以及为什么这么做。

In [ ]:
# Cell 1: who you are, and what system you chose
# 单元格 1：你是谁，你选了什么体系
#
# Fill in all four values, then run the cell. It writes system.txt, which the
# calculation cells below will read, so you only declare your system once.
# 填写下面四个值，然后运行。它会生成 system.txt，后面的计算单元格都从这个
# 文件读取，所以你只需要在这里声明一次。

NAME      = "____"   # your name / 你的名字 (pinyin or Chinese 拼音或中文)
USERNAME  = "____"   # your cluster username / 你的集群用户名
METAL     = "____"   # your FCC metal / 你的FCC金属: Cu, Ag, Pt, Pd, Al
ADSORBATE = "____"   # your adsorbate / 你的吸附物: H, N, CO, H2O, NH3 ...

# Look up the EXPERIMENTAL lattice constant of your metal in Angstrom and put
# it here (state your source in the markdown cell at the end of Part 1).
# 查一下你的金属的实验晶格常数（单位：埃 Angstrom），填在这里
# （在第 1 部分末尾的文字单元格里注明来源）。
A0_GUESS = ____      # e.g. 3.61 for Cu / 例如 Cu 是 3.61

# ---- checks: do not edit below / 检查：以下不要改 ----
assert NAME != "____" and USERNAME != "____", \
    "Fill in NAME and USERNAME / 请填写名字和用户名"
assert METAL in ("Cu", "Ag", "Pt", "Pd", "Al"), \
    "METAL must be one of Cu/Ag/Pt/Pd/Al (NOT Au) / 金属必须是 Cu/Ag/Pt/Pd/Al（不能是金）"
assert ADSORBATE not in ("O", "o", "Au"), \
    "Atomic O is the worked example: choose your own / 氧原子是课堂示例：请选你自己的"
assert 2.0 < float(A0_GUESS) < 6.0, \
    "A0_GUESS looks wrong: lattice constants are a few Angstrom / A0_GUESS 不对：晶格常数应为几个埃"

with open("system.txt", "w") as f:
    f.write(f"{METAL} {ADSORBATE} {A0_GUESS}\n")
print(f"OK: {NAME} ({USERNAME}) will study {ADSORBATE} on {METAL}(100). system.txt written.")

## Part 0: Setup / 准备工作

Run this notebook **from your own home directory**, not from `classfiles`
(you cannot write there, and every job writes its output next to this
notebook). One directory per study.

请在**你自己的主目录**下运行这个 notebook，不要在 `classfiles` 里运行
（那里你没有写权限，而且每个任务都会把输出写在 notebook 所在文件夹）。
请为每一个研究创建一个目录，并在该目录下进行研究。

The cell below just checks your environment is sane. It should print three
OK lines. If it does not, ask before continuing.

下面的单元格只是检查你的环境是否正常。它应该打印三行 OK。
如果没有，先提问再继续。

In [ ]:
# Cell 2: environment checks (nothing to fill in here)
# 单元格 2：环境检查（这里不需要填空）
import os, pathlib

here = pathlib.Path.cwd()
assert "classfiles" not in str(here), \
    "You are inside classfiles: copy this notebook to your home directory first / 你在 classfiles 里：请先把 notebook 复制到自己的主目录"
print(f"OK: working in {here}")

kw = pathlib.Path.home() / ".config" / "ase" / "castep_keywords.json"
print(f"OK: keyword cache present: {kw.exists()}")

import ase
print(f"OK: ase {ase.__version__}")

## Part 1: The Bulk Metal / 体相金属

Before you can trust a surface, you must converge and optimise the bulk
crystal it is cut from. Three steps, exactly as in the lectures:

在研究表面之前，必须先把体相晶体收敛并优化好。三步，和课上讲的一样：

1. **Cutoff energy** 截断能: how many plane waves the basis contains. (基组中有多少平面波)
2. **K-points** k点: how finely the Brillouin zone is sampled. (布里渊区采样有多密)
3. **Lattice constant** 晶格常数: the equilibrium cell size in DFT (not the
   experimental one: the two must not be mixed). (DFT计算后的平衡晶格常数(不是实验值，两者不能混用))


### 1.1 Cutoff convergence / 截断能收敛

The cell below submits **one Slurm job** that runs a series of single-point
calculations at increasing cutoff. You choose the series. Think: too coarse
a range tells you nothing; too fine wastes hours of queue time. Start around
300 eV; metals rarely need more than 700 eV (but *check*, don't believe me).

下面的单元格用于提交**一个 Slurm 任务**(计算任务)，在一系列递增的截断能下做单点计算。
这个序列由你来选。想一想：范围太粗看不出规律；太细浪费排队时间。
可以从 300 eV 左右开始；金属很少需要超过 700 eV（但要自行**验证**，
不要盲目相信我说的）。

In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J cutoff_conv
#!/usr/bin/python3
# Job 1.1: cutoff convergence sweep / 截断能收敛扫描
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import bulk
from ase.calculators.castep import Castep

metal, adsorbate, a0 = open('system.txt').read().split()
a0 = float(a0)

# ---- FILL IN / 填空 --------------------------------------------------------
# A list of cutoff energies in eV to test, from low to high.
# 要测试的截断能列表（单位 eV），从低到高。
cutoffs = ____        # e.g. / 例如 list(range(300, 751, 50))
# ---------------------------------------------------------------------------

atoms = bulk(metal, 'fcc', a=a0, cubic=True)

with open('results_cutoff.txt', 'w') as f:
    for c in cutoffs:
        calc = Castep(directory=f'bulk_cut_{c}', label=f'bulk_cut_{c}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        # ---- FILL IN / 填空: which parameter sets the cutoff? --------------
        # (look in the class notebook Part 2 / 参考课堂 notebook 第 2 部分)
        calc.param.____ = c
        # --------------------------------------------------------------------
        calc.param.task = 'SinglePoint'
        calc.cell.kpoint_mp_grid = '6 6 6'   # fixed while we test the cutoff
                                             # 测截断能时 k 点需保持不变
        atoms.calc = calc
        e = atoms.get_potential_energy()
        n = len(atoms)
        f.write(f"{c} {e/n}\n")
        f.flush()
        print(f"cutoff {c} eV -> {e/n:.6f} eV/atom", flush=True)
print("done")

### Analyse the cutoff sweep / 分析截断能扫描

Plot energy per atom against cutoff, then look at the **differences**
between successive points: convergence means the differences become small,
not that the curve looks flat to the eye. A common criterion: successive
energies within a few meV/atom.

画出每原子能量随截断能的变化，然后看相邻两点的**差值**：收敛的标准是
差值变小，而不是曲线"看起来平"。常用判据：相邻能量差在几个 meV/原子以内。

In [ ]:
# Cell 1.1b: read the results and decide / 读取结果并做判断
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('results_cutoff.txt')
cut, e = data[:, 0], data[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(cut, e, 'o-')
ax1.set_xlabel('cutoff (eV)')
ax1.set_ylabel('energy (eV/atom)')

# ---- FILL IN / 填空 --------------------------------------------------------
# Compute the difference between successive energies, in meV/atom.
# 计算相邻两个能量的差值，单位 meV/原子。
# Hint / 提示: np.diff(...) gives successive differences; 1 eV = 1000 meV.
de_meV = ____
# ---------------------------------------------------------------------------

ax2.semilogy(cut[1:], np.abs(de_meV), 'o-')
ax2.axhline(5, ls='--', c='r', label='5 meV/atom')
ax2.set_xlabel('cutoff (eV)')
ax2.set_ylabel('|dE| (meV/atom)')
ax2.legend()
plt.tight_layout()
for c2, d in zip(cut[1:], de_meV):
    print(f"{c2:6.0f} eV: change {d:+8.3f} meV/atom")

**My chosen cutoff / 我选择的截断能:** ____ eV

**Because / 理由:** ____ (state the criterion it satisfies
/ 说明它满足什么判据)

### 1.2 K-point convergence / k点收敛

Same idea, other knob: fix the cutoff at your chosen value, vary the
Monkhorst--Pack mesh. Metals have a Fermi surface, so they need denser
sampling than insulators: do not be surprised if this converges more slowly
than the cutoff did.

One thing to understand *now*, because it matters for the slab later: what
must stay consistent between different cells is the **linear density** of
k-points (points per unit length of reciprocal space), not the mesh label.
A 6×6×6 mesh in a small cell and a 6×6×6 mesh in a long cell are *not*
equivalent samplings.

同样的思路，换一个调节参数：把截断能固定在你选的值，逐步改变 Monkhorst-Pack 网格。
金属存在费米面，所以需要比绝缘体更密的采样：如果它比截断能收敛得慢，
不要惊讶。

现在就要理解一件事，因为这对后面构建slab(表面板模型)很重要：不同晶胞之间要保持一致的是
k点的**线密度**（倒空间单位长度里的点数），而不是网格的标签。
小晶胞里的 6×6×6 网格和长晶胞里的 6×6×6 网格，其采样密度**不是**等价的。

In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J kpt_conv
#!/usr/bin/python3
# Job 1.2: k-point convergence sweep / k点收敛扫描
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import bulk
from ase.calculators.castep import Castep

metal, adsorbate, a0 = open('system.txt').read().split()
a0 = float(a0)

# ---- FILL IN / 填空 --------------------------------------------------------
# The cutoff you chose in 1.1, and a list of mesh sizes k to test (the mesh
# will be k x k x k). / 你在 1.1 选的截断能，以及要测试的网格尺寸 k 的列表
# （网格为 k x k x k）。
CUTOFF = ____         # your converged cutoff / 你的收敛截断能 (eV)
kmeshes = ____        # e.g. / 例如 [4, 6, 8, 10, 12]
# ---------------------------------------------------------------------------

atoms = bulk(metal, 'fcc', a=a0, cubic=True)

with open('results_kpoints.txt', 'w') as f:
    for k in kmeshes:
        calc = Castep(directory=f'bulk_k_{k}', label=f'bulk_k_{k}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'SinglePoint'
        # ---- FILL IN / 填空: set the k x k x k mesh -------------------------
        # 提示 hint: the cutoff job fixed it as a string '6 6 6'
        calc.cell.kpoint_mp_grid = ____
        # --------------------------------------------------------------------
        atoms.calc = calc
        e = atoms.get_potential_energy()
        f.write(f"{k} {e/len(atoms)}\n")
        f.flush()
        print(f"mesh {k}x{k}x{k} -> {e/len(atoms):.6f} eV/atom", flush=True)
print("done")

In [ ]:
# Cell 1.2b: analyse (this one is complete: you saw the pattern in 1.1b)
# 单元格 1.2b：分析（这个是完整的：你在 1.1b 已经见过这个套路）
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('results_kpoints.txt')
k, e = data[:, 0], data[:, 1]
de_meV = np.diff(e) * 1000

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(k, e, 'o-')
ax1.set_xlabel('mesh k (k x k x k)')
ax1.set_ylabel('energy (eV/atom)')
ax2.semilogy(k[1:], np.abs(de_meV), 'o-')
ax2.axhline(5, ls='--', c='r', label='5 meV/atom')
ax2.set_xlabel('mesh k')
ax2.set_ylabel('|dE| (meV/atom)')
ax2.legend()
plt.tight_layout()
for k2, d in zip(k[1:], de_meV):
    print(f"{int(k2)}x{int(k2)}x{int(k2)}: change {d:+8.3f} meV/atom")

**My chosen mesh / 我选择的网格:** ____ × ____ × ____

**Question / 问题:** your bulk cell has side $a_0 \approx$ ____ Å. Later
your slab cell will be the same in $x, y$ but roughly 30 Å in $z$. Using the
*linear density* idea, what mesh should the slab use? Write the reasoning in
one or two sentences.

你的体相晶胞边长约为 ____ Å。之后的表面晶胞在 x、y 方向相同，但 z 方向约
30 Å。用**线密度**的思路，表面晶胞应该用什么网格？用一两句话写出理由。

**Answer / 回答:** ____

### 1.3 The lattice constant / 晶格常数

You have been building cells with the *experimental* lattice constant. DFT
with PBE has its own equilibrium: usually a percent or so larger. Everything
downstream (the slab, the adsorption geometry) must be built from the **DFT
value**, or forces will be lying to you. A geometry optimisation with a
variable cell finds it.

你此前都是按照实验晶格常数搭建计算原胞。而采用 PBE 泛函的密度泛函理论（DFT）计算会得出一套自身平衡晶格常数，通常比实验值偏大 1% 左右。后续所有建模体系（表面 slab、吸附构型）都必须以 DFT 自洽平衡晶格常数为基准搭建，否则计算得到的受力结果会完全失真。
可通过**可变晶胞几何优化**流程得到该 DFT 平衡晶格常数。

In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J latt_opt
#!/usr/bin/python3
# Job 1.3: optimise the lattice constant / 优化晶格常数
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import bulk
from ase.calculators.castep import Castep

metal, adsorbate, a0 = open('system.txt').read().split()
a0 = float(a0)

CUTOFF = ____         # same as 1.2 / 和 1.2 相同
KMESH  = ____         # your converged mesh as a string / 你的收敛网格，字符串
                      # e.g. / 例如 '8 8 8'

atoms = bulk(metal, 'fcc', a=a0, cubic=True)
calc = Castep(directory='bulk_opt', label='bulk_opt',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.cell.kpoint_mp_grid = KMESH

# ---- FILL IN / 填空 --------------------------------------------------------
# Which CASTEP task optimises the geometry? (The class notebook Part 4 uses
# it; single points were task='SinglePoint'.)
# 哪个 CASTEP task 会优化几何结构？（课堂 notebook 第 4 部分用过；
# 单点计算是 task='SinglePoint'。）
calc.param.____ = '____'
# ---------------------------------------------------------------------------

atoms.calc = calc
e = atoms.get_potential_energy()

# After the optimisation, read the relaxed cell back and save a0 for Part 2.
# 优化后，读取弛豫的晶胞，把 a0 存给第 2 部分用。
import numpy as np
from ase.io import read
final = read('bulk_opt/bulk_opt-out.cell')
a_opt = final.cell.lengths()[0]
with open('a0.txt', 'w') as f:
    f.write(f"{a_opt}\n")
print(f"optimised a0 = {a_opt:.4f} A (guess was {a0} A)")

**My DFT lattice constant / 我的 DFT 晶格常数:** ____ Å

**Experimental value and its source / 实验值及来源:** ____

**Comment / 评论:** is the deviation typical of PBE? (One sentence.)
偏差符合 PBE 的一般表现吗？（一句话。）____

## Part 2: The Clean Surface / 干净表面

A surface in a periodic code is a **slab**: a few layers of metal, a vacuum
gap, repeated forever. Two boundary tricks make it honest: the **bottom
layers are frozen** at bulk positions (pretending the crystal continues
below), and the **vacuum is wide enough** that the slab cannot see its own
periodic image.

周期性程序里的表面是一个**平板 (slab)**：几层金属加一段真空，无限重复。
有两种边界处理技巧使其能够真实地模拟表面：**底部原子层被冻结**在体相位置（以此模拟晶体在下方无限延伸），**真空足够宽**，平板"看不到"自己的周期镜像。

**Read this warning twice / 这个警告请读两遍：** in ASE's `fcc100`, layer
tags count from the **top**: tag 1 is the *surface* layer, the largest tag is
the *bottom*. Freeze the wrong side and your "surface" is a frozen wall while
the bulk-side atoms wave around: the numbers will look plausible and be
meaningless. (A previous year's notebook had exactly this bug.)

在 ASE 的 `fcc100` 里，层的标签 (tag) 是从**顶部**开始编号的：tag 1 是
**表面**层，最大的 tag 是**底部**。如果冻结了错误的一侧，你的"表面"将变成一堵冻结的墙，
而体相一侧的原子在乱动：计算结果可能看似合理，但物理意义是错误的。
（去年的一个 notebook 正好有这个 bug。）

In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 8 -J slab_layers
#!/usr/bin/python3
# Job 2.1: layer convergence via the surface energy / 用表面能做层数收敛
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import fcc100
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())     # from Part 1.3 / 来自 1.3
                                      # (if this fails: run 1.3 first / 先运行 1.3)

CUTOFF = ____         # your cutoff / 你的截断能
KMESH  = '____'       # slab mesh from your linear-density answer in 1.2
                      # 用你在 1.2 回答的线密度网格, e.g. '8 8 1'

# ---- FILL IN / 填空 --------------------------------------------------------
# Layer counts to test, and the vacuum gap in Angstrom. The assignment sheet
# suggests sensible starting points. / 要测试的层数列表和真空宽度（埃）。
# 作业说明里有合理的起点建议。
layer_list = ____     # e.g. / 例如 [3, 4, 5, 6, 7]
VACUUM = ____         # e.g. / 例如 12 (this is HALF the gap per side in ASE:
                      # fcc100's vacuum= is added above AND below / ASE 的
                      # vacuum= 会在上下各加这么多)
# ---------------------------------------------------------------------------

with open('results_layers.txt', 'w') as f:
    for nl in layer_list:
        slab = fcc100(metal, size=(1, 1, nl), a=a0, vacuum=VACUUM)
        # Freeze the bottom half. REMEMBER: tag 1 = TOP (surface).
        # 冻结下半部分。记住：tag 1 = 顶部（表面）。
        # ---- FILL IN / 填空: which atoms should be frozen? -----------------
        # We want atoms whose LAYER TAG is greater than half the layer count.
        # 我们要冻结"层标签大于层数一半"的原子。
        frozen = [atom.index for atom in slab if atom.tag ____ nl / 2]
        # --------------------------------------------------------------------
        slab.set_constraint(FixAtoms(indices=frozen))
        calc = Castep(directory=f'slab_{nl}L', label=f'slab_{nl}L',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'GeometryOptimization'
        calc.cell.kpoint_mp_grid = KMESH
        slab.calc = calc
        e = slab.get_potential_energy()
        area = slab.cell[0, 0] * slab.cell[1, 1]
        f.write(f"{nl} {e} {len(slab)} {area}\n")
        f.flush()
        print(f"{nl} layers: E = {e:.6f} eV ({len(frozen)} atoms frozen)",
              flush=True)
print("done")

### The surface energy / 表面能

The raw slab energy grows every time you add a layer, so it can never
converge. The quantity that *does* converge is the **surface energy**:

$$\gamma = \frac{E_{\text{slab}} - N\,E_{\text{bulk}}}{2A}$$

where $N$ is the number of atoms in the slab, $E_{\text{bulk}}$ the energy
**per atom** of your optimised bulk, and $A$ the surface area of the cell.
Why $2A$? Ask yourself how many surfaces a slab has. When $\gamma$ stops
changing with thickness, the slab middle has become "bulk-like": that is
your layer count.

平板的总能量每加一层都会变大，永远不会收敛。真正收敛的量是**表面能** γ：
其中 N 是平板里的原子数，E_bulk 是优化后体相的**每原子**能量，A 是晶胞的
表面面积。为什么除以 2A？想想一个平板有几个表面。当 γ 不再随厚度变化时，
平板中部已经"像体相了"：那就是你的层数。

In [ ]:
# Cell 2.1b: compute and plot the surface energy / 计算并画出表面能
import numpy as np
import matplotlib.pyplot as plt

# Energy per atom of YOUR optimised bulk, from bulk_opt (Part 1.3).
# 你优化后体相的每原子能量，来自 1.3 的 bulk_opt。
# Hint / 提示: the final energy is printed at the end of
# bulk_opt/bulk_opt.castep; the cubic fcc cell contains 4 atoms.
# 最终能量在 bulk_opt/bulk_opt.castep 的末尾；立方 fcc 晶胞含 4 个原子。
E_BULK_PER_ATOM = ____

data = np.loadtxt('results_layers.txt')
nl, e_slab, n_atoms, area = data.T

# ---- FILL IN / 填空: the surface energy formula ---------------------------
# eV/A^2 first; the conversion to J/m^2 is given below.
# 先算 eV/A^2；换算成 J/m^2 的代码在下面给出。
gamma_eV = ____
# ---------------------------------------------------------------------------

gamma_J = gamma_eV * 16.0218   # 1 eV/A^2 = 16.0218 J/m^2

plt.plot(nl, gamma_J, 'o-')
plt.xlabel('layers')
plt.ylabel('surface energy (J/m$^2$)')
plt.title('When this flattens, the slab is thick enough')
for n, g in zip(nl, gamma_J):
    print(f"{int(n)} layers: gamma = {g:.3f} J/m^2")

**My chosen thickness / 我选择的层数:** ____ layers

**My converged $\gamma$ / 我的收敛表面能:** ____ J/m²

**Sanity check / 合理性检查:** look up a typical experimental or computed
surface energy for your metal's (100) face. Same order of magnitude?
(One sentence, with the source.)
查一下你的金属 (100) 面的典型表面能（实验或计算值）。数量级一致吗？
（一句话，注明来源。）____

## From here, you are on your own / 从这里开始，需要靠你自己独立完成了

The scaffold ends here: **deliberately**. You now have a converged,
optimised bulk and a trustworthy clean surface, and you have practised every
skill the rest of the assignment needs: writing a job cell, sweeping a
parameter, reading results files, filling in a formula, and justifying a
choice in words.

模板到此结束：**这是有意设计的**。你现在有了收敛且优化好的体相、可靠的干净
表面模型，而且已经练习了剩余部分需要的全部技能：写任务单元格、扫描参数、
读结果文件、填公式、用文字说明你的选择。

What remains (see the assignment sheet for the marks):
剩下的部分（分值见作业说明）：

* **Part 3, the adsorbate reference / 吸附物参考态**: your species alone in
  a big box. Get the spin state right: the class notebook's O$_2$ is the
  cautionary tale. Then test *whose* cutoff matters. /
  你的吸附物单独放在大盒子里。自旋态一定要对：课堂 notebook 里的 O₂ 例子说明，自旋态的设置很   重要。然后测试到底是谁的截断能说了算。
* **Part 4, adsorption / 吸附**: on-top, bridge and hollow sites on your
  (100) slab, adsorption energies against the correct reference, preferred
  site and geometry. / 在 (100) 表面的 on-top、bridge、hollow 三个位点，
  用正确的参考态算吸附能，找出最稳定位点和吸附构型。
* **Part 5, notes and synthesis / 笔记与总结**: the summary table and four
  discussion questions, answered for *your* system. /
  总结表格和四道讨论题，根据**你自己的**体系回答。
* **Part 6 (optional) / 选做**: extensions, up to 10 marks. / 扩展题，最多加 10 分。

Build those cells the way this notebook builds its own: a `%%sbatch` header,
`module('load', 'compchem/castep')`, a calculator with your converged
settings, results written to a file next to the notebook. The class notebook
shows the *method* for every one of these steps on its own system. Understand
it, then write yours. Do not paste it: I will know.

照这个 notebook 的方式去写那些单元格：`%%sbatch` 开头、加载 castep 模块、
用你收敛好的参数进行计算、将结果写到 notebook 旁边的文件里。课堂的 notebook
对每一步的**方法**都有示范（用的是它自己的体系）。看懂它，然后写你自己的。
**不要直接粘贴：我看得出来。**

Submit everything to `~/submission/surface_study/` before the deadline
announced in the class group. Good luck: start the queue early, and read
your error messages: they are trying to help you.

在班级群公布的截止时间前，把所有内容提交到 `~/submission/surface_study/`。
祝顺利：另外请尽早提交计算任务，避免排队时间影响进度。认真阅读报错信息，它们通常能帮助你找到问题所在。